# Why Word Clouds Mislead

**Computational Text Analysis Pack · Module 10 — Visualising Text**

*Technique:* Diagnosing a visualisation; area perception and the alternatives

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/10b_why_word_clouds_mislead.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The word cloud is the default image of digital humanities, and it is the weakest
figure in the field. This notebook shows you exactly why — by building one and
then measuring what a reader can and cannot recover from it.

This is not snobbery. A word cloud can be the right choice for a poster or a
first look. The problem is using one where a comparison is being claimed.

## The tooling

**matplotlib**, and the argument that the chart is a claim. **seaborn** and
**ggplot2** (R) are the usual higher-level layers; the decisions this notebook
is about — axis, baseline, ordering — are yours regardless of library.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
LANGUAGE = "English"        # ← change this

sub = df[df["language"] == LANGUAGE]
counts = Counter()
for t in sub["text"]:
    counts.update(tokenise(t))
stop = {w for w, _ in counts.most_common(50)}
content = {w: n for w, n in counts.items() if w not in stop and n > 50}
top = dict(sorted(content.items(), key=lambda kv: -kv[1])[:45])
print(f"{len(top)} words, counts from {min(top.values()):,} to {max(top.values()):,}")

In [ ]:
# A word cloud, built from scratch — no library needed, and building it
# yourself makes the encoding visible.
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(10, 6))
vals = np.array(list(top.values()), dtype=float)
sizes = 8 + 34 * (vals - vals.min()) / (vals.max() - vals.min())
placed = []
for (word, n), size in zip(top.items(), sizes):
    for _ in range(200):
        x, y = rng.uniform(0.02, 0.98), rng.uniform(0.05, 0.95)
        w, h = len(word) * size * 0.0045, size * 0.012
        box = (x, y, x + w, y + h)
        if all(box[2] < p[0] or box[0] > p[2] or box[3] < p[1] or box[1] > p[3]
               for p in placed):
            placed.append(box)
            ax.text(x, y, word, fontsize=size, alpha=0.85)
            break
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("A word cloud")
plt.tight_layout(); plt.show()

## Now answer three questions from the picture

Do not scroll. Look at the cloud above and answer:

1. Which is the **third** most frequent word?
2. Is the largest word **twice** the size of the fifth, or five times?
3. Which two words have **almost the same** frequency?

You cannot answer any of them, and neither can your reader.

In [ ]:
ranked = sorted(top.items(), key=lambda kv: -kv[1])
print("The answers the cloud could not give you:")
print(f"  3rd most frequent : {ranked[2][0]} ({ranked[2][1]:,})")
print(f"  1st vs 5th        : {ranked[0][1] / ranked[4][1]:.1f}× "
      f"({ranked[0][1]:,} vs {ranked[4][1]:,})")
near = min(((a, b) for i, a in enumerate(ranked[:20]) for b in ranked[i + 1:20]),
           key=lambda p: abs(p[0][1] - p[1][1]))
print(f"  nearly equal      : {near[0][0]} ({near[0][1]:,}) and "
      f"{near[1][0]} ({near[1][1]:,})")

In [ ]:
# The same data, as a chart. Every question above is now answerable.
fig, ax = plt.subplots(figsize=(8, 8))
show = ranked[:30][::-1]
ax.barh([w for w, _ in show], [n for _, n in show])
ax.set_xlabel("occurrences")
ax.set_title("The same 30 words, sorted")
plt.tight_layout(); plt.show()

## The four specific faults

1. **Area is a poor visual encoding.** People systematically underestimate area
   differences; length on a common scale is the most accurate encoding there is.
2. **Font size means different things for different words.** A long word at size
   20 occupies far more ink than a short one at size 20, so *government* looks
   more important than *war* at identical frequency.
3. **Position is random and reads as meaningful.** Central and top words look
   important. They are wherever the layout algorithm put them.
4. **No axis, so no scale.** Nothing in the image tells you whether the largest
   word occurred 200 times or 200,000.

In [ ]:
# Fault 2, measured: ink is length × size, not size.
print(f"{'word':<16}{'count':>9}{'font size':>11}{'approx ink':>12}")
print("-" * 48)
for (word, n), size in list(zip(top.items(), sizes))[:10]:
    print(f"{word:<16}{n:>9,}{size:>11.1f}{len(word) * size:>12.0f}")
print()
lengths = np.array([len(w) for w in top])
r = np.corrcoef(lengths * sizes, vals)[0, 1]
r_size = np.corrcoef(sizes, vals)[0, 1]
print(f"correlation of FONT SIZE with frequency : {r_size:+.3f}")
print(f"correlation of INK AREA with frequency  : {r:+.3f}")
print()
print("Font size encodes frequency faithfully. Ink — what the eye actually")
print("responds to — does not, because word length gets mixed in.")

### 🔧 Your turn 1

Build word clouds for two categories side by side and try to say which words
differ between them.

This is the use that appears most in published work and it is the least
supportable: comparing two clouds by eye is impossible, because each is scaled to
its own maximum. Use the keyness scatter from 10a instead.

## When a word cloud is fine

- A poster or slide where the message is "here is roughly what this corpus is about"
- A first look at a corpus you have never opened
- Teaching, as a way into the idea that frequency varies

And even then, put the top-20 table next to it. That costs nothing and makes the
figure checkable.

### 🔧 Your turn 2

Take any word cloud from a published DH paper or a conference poster and write
down the claim its caption makes.

Then ask whether the image supports that claim. Usually the caption asserts a
comparison the picture cannot show — and the top-20 table would have shown it.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Side-by-side clouds are unreadable for comparison because each
is normalised to its own largest word. A word that is genuinely twice as common in
corpus A can appear the same size in both. The correct figure for "which words
differ" is 10a's keyness scatter or a paired bar chart of rates; the cloud belongs
to "what is this corpus about", which is a different question.

**Your turn 2.** The frequent pattern is a caption claiming a difference or a
trend, over an image that can only support "these words occur". Say what you can
support. "The 25 most frequent content words, after removing the 50 commonest
tokens" is a caption a word cloud can carry — and notice it also tells the reader
about the removal step, which most captions leave out.

</details>

## Where this stops

You can tell an honest text figure from a decorative one. Next: **10c**, on
showing where in a text something happens rather than how often.